In [1]:
"""
╔══════════════════════════════════════════════════════════════════╗
║  SCRIPT 2 — LightGBM + ELO  │  Predict 2023 NCAA Tournament     ║
║  Train : 2013–2022  (regular season stats + tournament labels    ║
║           + ELO ratings computed game-by-game)                   ║
║  Test  : 2023 Tournament (held-out, zero leakage)                ║
╚══════════════════════════════════════════════════════════════════╝

ELO Implementation
──────────────────
• Each team starts every season at ELO = 1500 (fresh-start per season).
• ELO is updated CHRONOLOGICALLY through every regular-season game
  and every prior-season tournament game — never peeking at future results.
• The ELO snapshot captured at the END of the regular season (before
  the tournament starts) is used as a feature for that season's
  tournament matchups.
• For the test season (2023) ELO is built only from 2023 regular-season
  games — tournament outcomes are never touched.

K-Factor schedule
  |MOV| > 10 pts → K = 30  (blowout — stronger signal)
  otherwise      → K = 20  (close game)
"""

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import log_loss, roc_auc_score

# ─────────────────────────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────────────────────────
m_regular_season_detailed = pd.read_csv('/Users/shaurya/Downloads/MRegularSeasonDetailedResults.csv')
m_seeds                   = pd.read_csv('/Users/shaurya/Downloads/MNCAATourneySeeds.csv')
m_tourney_detailed_result = pd.read_csv('/Users/shaurya/Downloads/MNCAATourneyDetailedResults.csv')

print("Regular Season shape :", m_regular_season_detailed.shape)
print("Seeds shape          :", m_seeds.shape)
print("Tourney Results shape:", m_tourney_detailed_result.shape)

# ─────────────────────────────────────────────────────────────────
# 2. PARSE SEEDS
# ─────────────────────────────────────────────────────────────────
def parse_seed(s):
    return int(''.join(filter(str.isdigit, s)))

m_seeds['SeedNum'] = m_seeds['Seed'].apply(parse_seed)

# ─────────────────────────────────────────────────────────────────
# 3. ELO ENGINE
# ─────────────────────────────────────────────────────────────────
def expected_score(r_a, r_b):
    """Probability that team A beats team B given ratings."""
    return 1.0 / (1.0 + 10 ** ((r_b - r_a) / 400.0))

def k_factor(score_diff):
    """Larger K for blowouts (>10 pts)."""
    return 30 if abs(score_diff) > 10 else 20

def compute_elo_end_of_regular_season(reg_df, seasons, base_elo=1500):
    """
    Walk through every regular-season game chronologically within each season.
    Returns a dict: {(season, team_id): elo_at_end_of_regular_season}

    Each team resets to base_elo at the start of every season.
    """
    elo_snapshot = {}

    for season in sorted(seasons):
        games = reg_df[reg_df['Season'] == season].sort_values('DayNum').copy()
        ratings = {}  # team_id → current elo

        def get_elo(tid):
            return ratings.get(tid, base_elo)

        for _, g in games.iterrows():
            w, l = int(g['WTeamID']), int(g['LTeamID'])
            ws, ls = g['WScore'], g['LScore']

            r_w = get_elo(w)
            r_l = get_elo(l)

            e_w = expected_score(r_w, r_l)
            e_l = 1.0 - e_w
            k   = k_factor(ws - ls)

            ratings[w] = r_w + k * (1.0 - e_w)   # winner
            ratings[l] = r_l + k * (0.0 - e_l)   # loser

        # Snapshot ELO for every team that played this season
        for tid, elo in ratings.items():
            elo_snapshot[(season, tid)] = elo

    return elo_snapshot

# ─────────────────────────────────────────────────────────────────
# 4. REGULAR-SEASON TEAM STATS (per season, no leakage)
# ─────────────────────────────────────────────────────────────────
def compute_team_stats(reg_df, seasons):
    rows = reg_df[reg_df['Season'].isin(seasons)].copy()

    win = rows.rename(columns={
        'WTeamID':'TeamID','LTeamID':'OppID',
        'WScore':'Pts','LScore':'OppPts',
        'WFGM':'FGM','WFGA':'FGA','WFGM3':'FGM3','WFGA3':'FGA3',
        'WFTM':'FTM','WFTA':'FTA','WOR':'OR','WDR':'DR',
        'WAst':'Ast','WTO':'TO','WStl':'Stl','WBlk':'Blk','WPF':'PF',
    }).copy()
    win['Win'] = 1

    lose = rows.rename(columns={
        'LTeamID':'TeamID','WTeamID':'OppID',
        'LScore':'Pts','WScore':'OppPts',
        'LFGM':'FGM','LFGA':'FGA','LFGM3':'FGM3','LFGA3':'FGA3',
        'LFTM':'FTM','LFTA':'FTA','LOR':'OR','LDR':'DR',
        'LAst':'Ast','LTO':'TO','LStl':'Stl','LBlk':'Blk','LPF':'PF',
    }).copy()
    lose['Win'] = 0

    keep = ['Season','TeamID','Pts','OppPts','FGM','FGA','FGM3','FGA3',
            'FTM','FTA','OR','DR','Ast','TO','Stl','Blk','PF','Win']
    all_g = pd.concat([win[keep], lose[keep]], ignore_index=True)

    agg = all_g.groupby(['Season','TeamID']).agg(
        Games       =('Win','count'),
        WinRate     =('Win','mean'),
        AvgPts      =('Pts','mean'),
        AvgOppPts   =('OppPts','mean'),
        AvgFGM      =('FGM','mean'),
        AvgFGA      =('FGA','mean'),
        AvgFGM3     =('FGM3','mean'),
        AvgFGA3     =('FGA3','mean'),
        AvgFTM      =('FTM','mean'),
        AvgFTA      =('FTA','mean'),
        AvgOR       =('OR','mean'),
        AvgDR       =('DR','mean'),
        AvgAst      =('Ast','mean'),
        AvgTO       =('TO','mean'),
        AvgStl      =('Stl','mean'),
        AvgBlk      =('Blk','mean'),
        AvgPF       =('PF','mean'),
    ).reset_index()

    agg['FGPct']  = agg['AvgFGM']  / agg['AvgFGA'].replace(0, np.nan)
    agg['FG3Pct'] = agg['AvgFGM3'] / agg['AvgFGA3'].replace(0, np.nan)
    agg['FTPct']  = agg['AvgFTM']  / agg['AvgFTA'].replace(0, np.nan)
    agg['Margin'] = agg['AvgPts']  - agg['AvgOppPts']
    return agg.set_index(['Season','TeamID'])

# ─────────────────────────────────────────────────────────────────
# 5. BUILD MATCHUP FEATURES  (stats + ELO + seeds)
# ─────────────────────────────────────────────────────────────────
def build_matchup_features(tourney_df, stats_df, seeds_df, elo_map, seasons,
                            base_elo=1500):
    records = []
    for _, row in tourney_df[tourney_df['Season'].isin(seasons)].iterrows():
        season = row['Season']
        w_id, l_id = row['WTeamID'], row['LTeamID']

        if w_id < l_id:
            t_a, t_b, label = w_id, l_id, 1
        else:
            t_a, t_b, label = l_id, w_id, 0

        if (season, t_a) not in stats_df.index or (season, t_b) not in stats_df.index:
            continue

        sa = stats_df.loc[(season, t_a)]
        sb = stats_df.loc[(season, t_b)]

        # ELO at end of regular season
        elo_a = elo_map.get((season, t_a), base_elo)
        elo_b = elo_map.get((season, t_b), base_elo)
        elo_prob_a = expected_score(elo_a, elo_b)   # ELO win probability for A

        # Seeds
        s_a = seeds_df[(seeds_df['Season']==season) & (seeds_df['TeamID']==t_a)]
        s_b = seeds_df[(seeds_df['Season']==season) & (seeds_df['TeamID']==t_b)]
        seed_a = s_a['SeedNum'].values[0] if len(s_a) else np.nan
        seed_b = s_b['SeedNum'].values[0] if len(s_b) else np.nan

        records.append({
            'Season': season, 'TeamA': t_a, 'TeamB': t_b, 'Label': label,

            # ── ELO features ──────────────────────────────────────
            'ELO_A':        elo_a,
            'ELO_B':        elo_b,
            'ELO_Diff':     elo_a - elo_b,
            'ELO_ProbA':    elo_prob_a,        # model-ready win prob from ELO

            # ── Seed features ──────────────────────────────────────
            'SeedA':        seed_a,
            'SeedB':        seed_b,
            'SeedDiff':     seed_a - seed_b,

            # ── Difference features (A − B) ────────────────────────
            'WinRateDiff':  sa['WinRate']  - sb['WinRate'],
            'PtsDiff':      sa['AvgPts']   - sb['AvgPts'],
            'OppPtsDiff':   sa['AvgOppPts']- sb['AvgOppPts'],
            'MarginDiff':   sa['Margin']   - sb['Margin'],
            'FGPctDiff':    sa['FGPct']    - sb['FGPct'],
            'FG3PctDiff':   sa['FG3Pct']   - sb['FG3Pct'],
            'FTPctDiff':    sa['FTPct']    - sb['FTPct'],
            'ORDiff':       sa['AvgOR']    - sb['AvgOR'],
            'DRDiff':       sa['AvgDR']    - sb['AvgDR'],
            'AstDiff':      sa['AvgAst']   - sb['AvgAst'],
            'TODiff':       sa['AvgTO']    - sb['AvgTO'],
            'StlDiff':      sa['AvgStl']   - sb['AvgStl'],
            'BlkDiff':      sa['AvgBlk']   - sb['AvgBlk'],
            'PFDiff':       sa['AvgPF']    - sb['AvgPF'],
            'GamesDiff':    sa['Games']    - sb['Games'],

            # ── Raw Team A ─────────────────────────────────────────
            'A_WinRate': sa['WinRate'], 'A_AvgPts': sa['AvgPts'],
            'A_Margin':  sa['Margin'],  'A_FGPct':  sa['FGPct'],
            'A_FG3Pct':  sa['FG3Pct'], 'A_FTPct':  sa['FTPct'],
            'A_AvgOR':   sa['AvgOR'],   'A_AvgDR':  sa['AvgDR'],
            'A_AvgAst':  sa['AvgAst'], 'A_AvgTO':   sa['AvgTO'],
            'A_AvgStl':  sa['AvgStl'], 'A_AvgBlk':  sa['AvgBlk'],

            # ── Raw Team B ─────────────────────────────────────────
            'B_WinRate': sb['WinRate'], 'B_AvgPts': sb['AvgPts'],
            'B_Margin':  sb['Margin'],  'B_FGPct':  sb['FGPct'],
            'B_FG3Pct':  sb['FG3Pct'], 'B_FTPct':  sb['FTPct'],
            'B_AvgOR':   sb['AvgOR'],   'B_AvgDR':  sb['AvgDR'],
            'B_AvgAst':  sb['AvgAst'], 'B_AvgTO':   sb['AvgTO'],
            'B_AvgStl':  sb['AvgStl'], 'B_AvgBlk':  sb['AvgBlk'],
        })
    return pd.DataFrame(records)

# ─────────────────────────────────────────────────────────────────
# 6. TRAIN / TEST SPLIT
#    Train  : 2013–2022 tournament matchups
#    Test   : 2023 tournament matchups (fully held-out)
#
#    ELO is computed from regular-season games only, per season.
#    The 2023 ELO uses ONLY 2023 regular-season data.
# ─────────────────────────────────────────────────────────────────
TRAIN_SEASONS = list(range(2013, 2023))   # 2013 … 2022
TEST_SEASONS  = [2023]

all_seasons = TRAIN_SEASONS + TEST_SEASONS

# Build stats and ELO for ALL seasons (no cross-season leakage)
stats   = compute_team_stats(m_regular_season_detailed, all_seasons)
elo_map = compute_elo_end_of_regular_season(m_regular_season_detailed, all_seasons)

train_df = build_matchup_features(
    m_tourney_detailed_result, stats, m_seeds, elo_map, TRAIN_SEASONS)
test_df  = build_matchup_features(
    m_tourney_detailed_result, stats, m_seeds, elo_map, TEST_SEASONS)

print(f"\nTrain matchups : {len(train_df)}  (2013–2022 tournaments)")
print(f"Test  matchups : {len(test_df)}   (2023 tournament only)")

# Spot-check ELO values for 2023
print("\nSample ELO values (2023 regular-season end):")
elo_2023 = {k: v for k, v in elo_map.items() if k[0] == 2023}
top5 = sorted(elo_2023.items(), key=lambda x: -x[1])[:5]
for (s, t), e in top5:
    print(f"  Season {s} | Team {t} | ELO {e:.1f}")

# ─────────────────────────────────────────────────────────────────
# 7. FEATURE COLUMNS
# ─────────────────────────────────────────────────────────────────
FEATURE_COLS = [
    # ELO
    'ELO_A','ELO_B','ELO_Diff','ELO_ProbA',
    # Seeds
    'SeedA','SeedB','SeedDiff',
    # Difference
    'WinRateDiff','PtsDiff','OppPtsDiff','MarginDiff',
    'FGPctDiff','FG3PctDiff','FTPctDiff',
    'ORDiff','DRDiff','AstDiff','TODiff','StlDiff','BlkDiff','PFDiff','GamesDiff',
    # Raw A
    'A_WinRate','A_AvgPts','A_Margin','A_FGPct','A_FG3Pct','A_FTPct',
    'A_AvgOR','A_AvgDR','A_AvgAst','A_AvgTO','A_AvgStl','A_AvgBlk',
    # Raw B
    'B_WinRate','B_AvgPts','B_Margin','B_FGPct','B_FG3Pct','B_FTPct',
    'B_AvgOR','B_AvgDR','B_AvgAst','B_AvgTO','B_AvgStl','B_AvgBlk',
]

X_train = train_df[FEATURE_COLS]
y_train = train_df['Label']
X_test  = test_df[FEATURE_COLS]
y_test  = test_df['Label']

# ─────────────────────────────────────────────────────────────────
# 8. TRAIN LIGHTGBM
# ─────────────────────────────────────────────────────────────────
lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_valid = lgb.Dataset(X_test,  label=y_test, reference=lgb_train)

params = {
    'objective':         'binary',
    'metric':            'binary_logloss',
    'boosting_type':     'gbdt',
    'num_leaves':        31,
    'learning_rate':     0.05,
    'feature_fraction':  0.8,
    'bagging_fraction':  0.8,
    'bagging_freq':      5,
    'min_child_samples': 10,
    'lambda_l1':         0.1,
    'lambda_l2':         0.1,
    'verbose':           -1,
    'seed':              42,
}

model = lgb.train(
    params,
    lgb_train,
    num_boost_round=500,
    valid_sets=[lgb_valid],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(50)],
)

# ─────────────────────────────────────────────────────────────────
# 9. EVALUATE ON 2023 TOURNAMENT
# ─────────────────────────────────────────────────────────────────
y_pred_prob = model.predict(X_test)
y_pred_bin  = (y_pred_prob >= 0.5).astype(int)

ll  = log_loss(y_test, y_pred_prob)
auc = roc_auc_score(y_test, y_pred_prob)
acc = (y_pred_bin == y_test).mean()

print(f"\n{'='*50}")
print(f"  SCRIPT 2 — 2023 Tournament Evaluation  (LightGBM + ELO)")
print(f"{'='*50}")
print(f"  Log Loss  : {ll:.4f}")
print(f"  ROC AUC   : {auc:.4f}")
print(f"  Accuracy  : {acc:.4f}  ({int(acc*len(y_test))}/{len(y_test)} correct)")

# ─────────────────────────────────────────────────────────────────
# 10. FEATURE IMPORTANCE
# ─────────────────────────────────────────────────────────────────
importance = pd.DataFrame({
    'Feature':    model.feature_name(),
    'Importance': model.feature_importance(importance_type='gain'),
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print("\nTop 15 Features by Gain (ELO highlighted):")
print(importance.head(15).to_string(index=False))

# ─────────────────────────────────────────────────────────────────
# 11. PREDICTIONS OUTPUT
# ─────────────────────────────────────────────────────────────────
results_df = test_df[['Season','TeamA','TeamB','Label'] + FEATURE_COLS].copy()
results_df['PredProb_TeamA_Wins'] = y_pred_prob
results_df['PredWinner']   = results_df.apply(
    lambda r: r['TeamA'] if r['PredProb_TeamA_Wins'] >= 0.5 else r['TeamB'], axis=1)
results_df['ActualWinner'] = results_df.apply(
    lambda r: r['TeamA'] if r['Label'] == 1 else r['TeamB'], axis=1)
results_df['Correct']      = (results_df['PredWinner'] == results_df['ActualWinner']).astype(int)
results_df['FeaturesUsed'] = ' | '.join(FEATURE_COLS)

results_df.to_csv('/Users/shaurya/Downloads/script2_2023_predictions_elo.csv', index=False)
importance.to_csv('/Users/shaurya/Downloads/script2_feature_importance_elo.csv', index=False)

print(f"\n✅  Predictions  → /Users/shaurya/Downloads/script2_2023_predictions_elo.csv")
print(f"✅  Importance   → /Users/shaurya/Downloads/script2_feature_importance_elo.csv")
print(f"\nTotal features : {len(FEATURE_COLS)}")
print("Features list  :", FEATURE_COLS)

# ─────────────────────────────────────────────────────────────────
# 12. ELO-ONLY BASELINE  (how well does raw ELO predict alone?)
# ─────────────────────────────────────────────────────────────────
elo_pred_prob = test_df['ELO_ProbA'].values
elo_ll  = log_loss(y_test, elo_pred_prob)
elo_auc = roc_auc_score(y_test, elo_pred_prob)
elo_acc = ((elo_pred_prob >= 0.5).astype(int) == y_test.values).mean()

print(f"\n{'='*50}")
print(f"  ELO-ONLY BASELINE (for reference)")
print(f"{'='*50}")
print(f"  Log Loss  : {elo_ll:.4f}")
print(f"  ROC AUC   : {elo_auc:.4f}")
print(f"  Accuracy  : {elo_acc:.4f}  ({int(elo_acc*len(y_test))}/{len(y_test)} correct)")
print(f"\n  LightGBM+ELO gain over ELO-only baseline:")
print(f"  ΔLog Loss : {elo_ll - ll:+.4f}  (negative = LightGBM better)")
print(f"  ΔAUC      : {auc - elo_auc:+.4f}  (positive = LightGBM better)")

Regular Season shape : (122775, 34)
Seeds shape          : (2626, 3)
Tourney Results shape: (1449, 34)

Train matchups : 602  (2013–2022 tournaments)
Test  matchups : 67   (2023 tournament only)

Sample ELO values (2023 regular-season end):
  Season 2023 | Team 1222 | ELO 1747.4
  Season 2023 | Team 1104 | ELO 1736.0
  Season 2023 | Team 1417 | ELO 1725.7
  Season 2023 | Team 1194 | ELO 1723.1
  Season 2023 | Team 1266 | ELO 1714.2
[50]	valid_0's binary_logloss: 0.667781

  SCRIPT 2 — 2023 Tournament Evaluation  (LightGBM + ELO)
  Log Loss  : 0.6332
  ROC AUC   : 0.6963
  Accuracy  : 0.6269  (42/67 correct)

Top 15 Features by Gain (ELO highlighted):
   Feature  Importance
  SeedDiff  578.411579
MarginDiff  204.580419
     ELO_A  133.368549
 GamesDiff  121.760157
     SeedA  115.673218
  A_AvgStl  109.761406
   B_FGPct  100.855700
    ORDiff  100.340330
   A_AvgDR   99.535869
    DRDiff   94.803809
  A_AvgBlk   93.201007
  B_Margin   90.889159
  A_AvgPts   85.720543
    PFDiff   83.587